# User Comment Processing

Regenerate the per-topic phrase mapping table (sheet `1166121277`) using **all** curated phrases from the paired-comparison sheet (`1103808983`), incorporating newly added annotations in `CuratePhrasesMoreComplex`.

**Data flow:**
- Sheet `1103808983` → paired comparisons with `CuratePhrasesMoreComplex` / `CuratePhrasesLessComplex` (JSON dicts keyed by topic)
- Sheet `1166121277` → existing per-topic phrase table with `{topic}`, `{topic} (AIcurated)`, `{topic} (HumanCuration)`, `{topic}_sentiment`, `{topic}_count`
- Goal: rebuild the full topic phrase table from scratch using all curated phrases, retaining existing AI/HC mappings where available

### Execution guide

**Required:** cells 2-3, 5, 7, 9, 11-12, and 16. Cell 11 contains the manual mappings for phrases that do not yet have AI/Human-curated labels; review those mappings before generating the table.

**Human-curation checkpoint:** review the generated `AIcurated` and `HumanCuration` columns before refreshing `1.phraseCuration` (gid `1166121277`). Refresh `2.curatedPhrases` (gid `517808299`) with the reviewed reduction input afterward.

**Optional:** cell 14 only compares old and new tables. Cells 17-18 convert the result to the keyword-count format for `A.keyword.CountByVisTypes` (gid `1160817631`); run them only when that workbook summary is needed.

In [ ]:
# --- portable setup (added by clean-up) ---
import os, sys
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
os.makedirs('data_processing', exist_ok=True)
from data_loader import load_sheet, load_paired_comparison
import config


In [ ]:
import pandas as pd
import json
from collections import defaultdict, Counter

# ── Spreadsheet IDs ─────────────────────────────────────────────────────────────
SPREADSHEET_PAIRS = '1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc'
SPREADSHEET_ORDER = '1gfeYdT-RxLq9tvNpfv_P8AIKg2eSz6-8bSv5zUJsWEc'

# 7 canonical topics
TOPICS = [
    'Immediacy / Cognitive Load',
    'Data Density / Image Clutter',
    'Visual Encoding Clarity',
    'Semantics / Text Legibility',
    'Schema',
    'Color, Symbol, and Texture Details',
    'Aesthetics Uncertainty',
]

# ── Load paired-comparison sheet (1103808983) ────────────────────────────────────
df_pairs = load_sheet(SPREADSHEET_PAIRS, '1103808983')
print(f'Raw paired comparisons: {len(df_pairs)} rows')

# Filter out ToRemove rows
df_pairs = df_pairs[df_pairs['ToRemove'].fillna(0).astype(float) != 1.0].copy()
print(f'After removing ToRemove: {len(df_pairs)} rows')

# ── Load existing phrase mapping (1166121277) for AI/HC lookup ───────────────────
df_existing = load_sheet(SPREADSHEET_ORDER, '1166121277')
print(f'\nExisting term mapping: {df_existing.shape}')
print(f'Columns: {df_existing.columns.tolist()}')

## Step 1: Parse All Curated Phrases from Sheet 1103808983
Each `CuratePhrasesMoreComplex` / `CuratePhrasesLessComplex` cell contains a JSON dict keyed by topic name, e.g. `{"Schema": ["bar graph", "pie chart"], "Immediacy / Cognitive Load": ["hard to read"]}`.

In [ ]:
import re as _re

def parse_curated_json(json_str):
    """Parse a JSON string into a topic→phrases dict. Returns empty dict on failure.
    Handles trailing commas and stray extra quotes."""
    if not isinstance(json_str, str) or json_str.strip() == '':
        return {}
    try:
        d = json.loads(json_str)
        if isinstance(d, dict):
            return d
    except json.JSONDecodeError:
        pass
    # Fix trailing commas before ] or } and stray double-quotes
    fixed = _re.sub(r',\s*]', ']', json_str)
    fixed = _re.sub(r',\s*}', '}', fixed)
    fixed = _re.sub(r'"\s*"(\([+-]\))', r'"\1', fixed)  # ""(-) → "(-) 
    try:
        d = json.loads(fixed)
        if isinstance(d, dict):
            return d
    except json.JSONDecodeError:
        pass
    return {}

# ── Collect all (topic, phrase, sentiment) per image ────────────────────────────
topic_phrase_images = defaultdict(set)       # (topic, phrase) → {imageName, ...}
topic_phrase_sentiments = defaultdict(set)    # (topic, phrase) → {'(+)', '(-)'}

def extract_phrases_with_sentiment(phrases_list):
    """Given a list of phrases (possibly with (+)/(-) prefixes), return (bare_phrase, sentiment) pairs."""
    results = []
    if not isinstance(phrases_list, list):
        return results
    for p in phrases_list:
        if not isinstance(p, str) or not p.strip():
            continue
        p = p.strip()
        sentiment = ''
        if p.startswith('(+)'):
            sentiment = '(+)'
            p = p[3:].strip()
        elif p.startswith('(-)'):
            sentiment = '(-)'
            p = p[3:].strip()
        if p:
            results.append((p, sentiment))
    return results

# ── Column names (verified from actual sheet) ────────────────────────────────────
IMG_MORE_COL = 'moreComplexImageName'
IMG_LESS_COL = 'lessComplexImageName'
CURATE_MORE_COL = 'CuratePhrasesMoreComplex'
CURATE_LESS_COL = 'CuratePhrasesLessComplex'

# ── Map non-canonical topic names to canonical ones ──────────────────────────────
TOPIC_ALIASES = {
    'Density / Clutter': 'Data Density / Image Clutter',
    'Encoding Clarity': 'Visual Encoding Clarity',
}

n_parsed_more = 0
n_parsed_less = 0

for _, row in df_pairs.iterrows():
    # More complex side
    more_img = row.get(IMG_MORE_COL, '')
    curated_more = parse_curated_json(row[CURATE_MORE_COL])
    if curated_more:
        n_parsed_more += 1
    if isinstance(more_img, str) and more_img.strip():
        for topic, phrases in curated_more.items():
            topic = TOPIC_ALIASES.get(topic, topic)
            for phrase, sent in extract_phrases_with_sentiment(phrases):
                topic_phrase_images[(topic, phrase)].add(more_img.strip())
                if sent:
                    topic_phrase_sentiments[(topic, phrase)].add(sent)

    # Less complex side
    less_img = row.get(IMG_LESS_COL, '')
    curated_less = parse_curated_json(row[CURATE_LESS_COL])
    if curated_less:
        n_parsed_less += 1
    if isinstance(less_img, str) and less_img.strip():
        for topic, phrases in curated_less.items():
            topic = TOPIC_ALIASES.get(topic, topic)
            for phrase, sent in extract_phrases_with_sentiment(phrases):
                topic_phrase_images[(topic, phrase)].add(less_img.strip())
                if sent:
                    topic_phrase_sentiments[(topic, phrase)].add(sent)

# Stats
print(f'Rows with parseable JSON — More: {n_parsed_more}, Less: {n_parsed_less}')
all_topics_found = sorted(set(t for t, _ in topic_phrase_images))
print(f'Unique (topic, phrase) pairs: {len(topic_phrase_images)}')
print(f'Topics found in JSON:')
for t in all_topics_found:
    n = sum(1 for (tp, _) in topic_phrase_images if tp == t)
    print(f'  {t}: {n} unique phrases')

# Check for topic names that don't match our canonical list
unknown_topics = set(all_topics_found) - set(TOPICS)
if unknown_topics:
    print(f'\n⚠ Non-canonical topics found: {unknown_topics}')

## Step 2: Build Existing AI/HC Lookup from Sheet 1166121277

In [ ]:
# ── Build (topic, original_phrase) → (AIcurated, HumanCurated, sentiment) lookup ──
existing_lookup = {}  # (topic, phrase) → {'ai': str, 'hc': str, 'sentiment': str}

for topic in TOPICS:
    col_orig = topic
    col_ai   = f'{topic} (AIcurated)'
    col_hc   = f'{topic} (HumanCuration)'
    col_sent = f'{topic}_sentiment'

    for _, row in df_existing.iterrows():
        orig = row.get(col_orig)
        if not isinstance(orig, str) or orig.strip() == '':
            continue
        orig = orig.strip()
        ai = str(row.get(col_ai, '')).strip() if isinstance(row.get(col_ai), str) else ''
        hc = str(row.get(col_hc, '')).strip() if isinstance(row.get(col_hc), str) else ''
        sent = str(row.get(col_sent, '')).strip() if isinstance(row.get(col_sent), str) else ''
        existing_lookup[(topic, orig)] = {'ai': ai, 'hc': hc, 'sentiment': sent}

print(f'Existing lookup entries: {len(existing_lookup)}')

# How many of our extracted phrases match?
matched = sum(1 for k in topic_phrase_images if k in existing_lookup)
unmatched = len(topic_phrase_images) - matched
print(f'Matched to existing AI/HC mapping: {matched}')
print(f'New (no existing mapping): {unmatched}')

# Show sample of unmatched
unmatched_list = [(t, p) for (t, p) in topic_phrase_images if (t, p) not in existing_lookup]
if unmatched_list:
    print(f'\nSample new phrases without AI/HC mapping (first 15):')
    for t, p in sorted(unmatched_list)[:15]:
        n_imgs = len(topic_phrase_images[(t, p)])
        print(f'  [{t}] "{p}" ({n_imgs} images)')

## Step 3: Regenerate Per-Topic Phrase Table
Build the same wide-format table as sheet `1166121277`: for each topic, 5 columns (`_sentiment`, `_count`, original, `(AIcurated)`, `(HumanCuration)`), sorted by image count descending.

In [ ]:
# Collect existing AI and HC vocabularies per topic, and unmapped phrases
topic_ai_vocab = {}
topic_hc_vocab = {}
topic_unmapped = {}

for topic in TOPICS:
    entries = [(k[1], v) for k, v in existing_lookup.items() if k[0] == topic]
    topic_ai_vocab[topic] = sorted(set(v['ai'] for _, v in entries if v['ai']))
    topic_hc_vocab[topic] = sorted(set(v['hc'] for _, v in entries if v['hc']))
    unmapped = [(p, len(topic_phrase_images[(topic, p)])) 
                for (t, p) in topic_phrase_images if t == topic and (t, p) not in existing_lookup]
    topic_unmapped[topic] = sorted(unmapped, key=lambda x: -x[1])

# Dump to temp file for inspection
with open('data_processing/_unmapped_debug.txt', 'w', encoding='utf-8') as f:
    for topic in TOPICS:
        f.write(f'\n=== {topic} ===\n')
        f.write(f'AI vocab ({len(topic_ai_vocab[topic])}): {topic_ai_vocab[topic]}\n')
        f.write(f'HC vocab ({len(topic_hc_vocab[topic])}): {topic_hc_vocab[topic]}\n')
        f.write(f'Unmapped ({len(topic_unmapped[topic])}):\n')
        for p, cnt in topic_unmapped[topic]:
            f.write(f'  "{p}" ({cnt} imgs)\n')

total_unmapped = sum(len(v) for v in topic_unmapped.values())
print(f'Total unmapped: {total_unmapped} — details in data_processing/_unmapped_debug.txt')

### Step 3b: Fill AI/HC Mappings for Unmapped Phrases
For each unmapped phrase, assign the best-matching existing AIcurated and HumanCuration term from that topic's vocabulary. New terms are created only when no existing term is a good semantic match.

In [ ]:
# ── Manual AI/HC mappings for all 179 unmapped phrases ──────────────────────────
# Format: { topic: { original_phrase: (AIcurated, HumanCuration) } }
# Matches existing vocab where possible; new terms only when no good match exists.

NEW_MAPPINGS = {
    'Immediacy / Cognitive Load': {
        'Comprehensibility':
            ('easier to understand', 'easy to interpret/read/understand'),
        'How easily I could understand the information presented.':
            ('easier to understand', 'easier to interpret/read/understand'),
        'How quickly I realised what the information was showing.':
            ('understand at a glance', 'easy to interpret/read/understand'),
        'How much I had to read.':
            ('needs a lot of reading', 'more reading/interpretation/understanding'),
        'Initial (intuitive) attractiveness to actually delve into the image and analyze it':
            ('not intuitive', 'intuitive'),
        'Expected time and cognitive load to understand the logic of presentation':
            ('take longer to understand', 'take longer to interpret'),
        'Expected time and cognitive load to understand the key takeaway of/ key patterns suggested by the image (separating information from noise)':
            ('take longer to process', 'more effort/reading/detailed analysis'),
        'Expected time and cognitive load to understand the majority of information in the image (separating information from noise)':
            ('take longer to process', 'more effort/reading/detailed analysis'),
        'make sense.':
            ('easier to understand', 'easy to interpret/read/understand'),
        'more statistics':
            ('more data to comprehend', 'more to read/analyze/understand'),
        'comprehend the context and process the provided information.':
            ('complex to process', 'complicated to process'),
        'search for any correlation and significance in the available data.':
            ('need detailed analysis', 'more effort/reading/detailed analysis'),
        'The info is more varied.':
            ('more data to comprehend', 'more to read/analyze/understand'),
        'The info doesn\'t seem to follow a specific pattern.':
            ('confusing', 'unclear meaning/confusing'),
        'several aspects of the information can be interrelated':
            ('more factors to consider', 'more to read/analyze/understand'),
        'deductions can be taken':
            ('provokes thought', 'provokes thought and understanding'),
        'The ease of reading the image':
            ('easy to read', 'easy to interpret/read/understand'),
        'The time it takes to read the image':
            ('take longer to interpret', 'take longer to interpret'),
        'The level of attention it requires to read the image':
            ('requires concentration', 'attention/squinting to understand'),
        'easier to read and understand':
            ('easier to understand', 'easier to interpret/read/understand'),
        'unfamiliar terms':
            ('require specialized knowledge', 'require specialized knowledge'),
        'more impact':
            ('provokes thought', 'provokes thought and understanding'),
        'More complex colour scheme':
            ('more complicated', 'more complicated'),
        'Less linear':
            ('more intricate', 'more intricate'),
        'Less symetrical':
            ('more intricate', 'more intricate'),
        'It\'s more difficult to read the words because they are small;':
            ('hard to read', 'hard to interpret/read/understand'),
        'the graphic is vertical.':
            ('less straightforward', 'less intuitive'),
        'make it more complex than a 2d projection':
            ('more complicated', 'more complicated'),
        'Understanding what the visualization is about':
            ('difficult to understand', 'hard to interpret/read/understand'),
        'Understanding different value rankings':
            ('difficult to understand', 'hard to interpret/read/understand'),
        'Looks complicated':
            ('complicated', 'more complicated'),
        'a large number of variables (countries)':
            ('more factors to consider', 'more to read/analyze/understand'),
        'The data covers a large number of years.':
            ('more data to comprehend', 'more to read/analyze/understand'),
        'The image contains a greater variety of colours.':
            ('more complicated', 'more complicated'),
        'more information':
            ('more data to comprehend', 'more to read/analyze/understand'),
        'mixed the bar on the top is imposing':
            ('confusing', 'unclear meaning/confusing'),
        'the color are varied':
            ('more complicated', 'more complicated'),
        'Information about what the visualisation is showing':
            ('needs understanding', 'more reading/interpretation/understanding'),
        'really complex for me':
            ('complicated', 'complicated to process'),
        'i couldnt say what is complexer':
            ('not sure', 'not interpretable/readable/understandable'),
        'confusion':
            ('confusing', 'unclear meaning/confusing'),
        'readability':
            ('hard to read', 'hard to interpret/read/understand'),
        'Time to evaluate context':
            ('take longer to process', 'take longer to interpret'),
        'Volume of information':
            ('more data to comprehend', 'more to read/analyze/understand'),
        'Data interpretation':
            ('requires more interpretation', 'more reading/interpretation/understanding'),
        'Spread of information on the image':
            ('more data to comprehend', 'more to read/analyze/understand'),
        'Complexity':
            ('complicated', 'complicated to process'),
        'number of shapes':
            ('more factors to consider', 'more to read/analyze/understand'),
        'jumped out to me as instantly making it more complex.':
            ('more complicated', 'more complicated'),
        'Variety and Density of Elements':
            ('more factors to consider', 'more to read/analyze/understand'),
        'Level of Detail':
            ('need detailed analysis', 'more effort/reading/detailed analysis'),
        'Depth and Dimensionality':
            ('more intricate', 'more intricate'),
        'Interconnectedness of Elements':
            ('more intricate', 'more intricate'),
    },

    'Data Density / Image Clutter': {
        'It contains more information.':
            ('more information', 'much/more data/info/info spread'),
        'Variety and Density of Elements':
            ('information density', 'dense/cluttered data/info'),
        'Expected time and cognitive load to understand the key takeaway of/ key patterns suggested by the image (separating information from noise)':
            ('more to understand', 'more to read/analyze/understand'),
        'Expected time and cognitive load to understand the majority of information in the image (separating information from noise)':
            ('more to understand', 'more to read/analyze/understand'),
        'overlapping volume graphs':
            ('overlapping', 'overlapping shapes/colors/lines'),
        'multiple trends to compare to each other':
            ('multiple features', 'multiple features/elements/graphs'),
        'more detailed graphics':
            ('more detailed', 'more detailed/things'),
        'more statistics':
            ('more information', 'much/more data/info/info spread'),
        'Amount of data':
            ('more data', 'much/more data/info/info spread'),
        'More informations in this image.':
            ('more information', 'much/more data/info/info spread'),
        'It has more visual aids.':
            ('more elements', 'more charts/points/lines/shapes/elements'),
        'a lot of useful information':
            ('packed with information', 'much/more data/info/info spread'),
        'It has more visual data':
            ('more data', 'much/more data/info/info spread'),
        'It looks more visually complex.':
            ('more detail', 'more detailed/things'),
        'how many type of entities they were':
            ('many elements', 'many points/lines/shapes/elements'),
        'numbers on entities':
            ('many numbers', 'many numbers'),
        'Number of elements':
            ('many elements', 'many points/lines/shapes/elements'),
        'The number of graphic elements in each image':
            ('many elements', 'many points/lines/shapes/elements'),
        'a lot of color and text':
            ('more content', 'much/more data/info/info spread'),
        'All points converges to each other so that it makes more sense':
            ('clustered information', 'clustered data/info'),
        'the level of details that it has provide way to much information':
            ('too much information', 'too much data/info'),
        'Numbers of entities':
            ('many elements', 'many points/lines/shapes/elements'),
        'Information density':
            ('information density', 'dense/cluttered data/info'),
        'more info':
            ('more information', 'much/more data/info/info spread'),
        'a large number of variables (countries)':
            ('more variables', 'more parameters/variables'),
        'The data covers a large number of years.':
            ('larger dataset', 'large dataset'),
        'mixed the bar on the top is imposing':
            ('cluttered', 'dense/cluttered layout'),
        'Number of individual squares':
            ('many shapes', 'many points/lines/shapes/elements'),
        'Number of letters':
            ('many numbers', 'many numbers'),
        'Number of unique shapes':
            ('many shapes', 'many points/lines/shapes/elements'),
        'condensate':
            ('concentrated information', 'concentrated'),
        'confusion':
            ('cluttered', 'dense/cluttered data/info'),
        'simplicity':
            ('simple information', 'simple information'),
        'Spread of information on the image':
            ('more information spread', 'much/more data/info/info spread'),
        'Complexity':
            ('more detail', 'more detailed/things'),
        'The inclusion of connections between different points in the image':
            ('multiple interacting elements', 'multiple interacting elements'),
    },

    'Visual Encoding Clarity': {
        'Expected time and cognitive load to understand the logic of presentation':
            ('unclear structure', 'unclear structure'),
        'Expected time and cognitive load to understand the key takeaway of/ key patterns suggested by the image (separating information from noise)':
            ('unclear what data conveys', 'unclear what data conveys'),
        'Expected time and cognitive load to understand the majority of information in the image (separating information from noise)':
            ('unclear what data conveys', 'unclear what data conveys'),
        'set of geometry':
            ('shapes', 'shapes'),
        'alignment':
            ('alignment', 'shape arrangement'),
        'tell the story':
            ('information nuance expressed', 'information nuance expressed'),
        'graphs are simple':
            ('simple visual elements', 'simple visual elements'),
        'More different shapes':
            ('more shape variation', 'shape variation'),
        'compare images and determine which one is the closest representation of the concept they are trying to convey.':
            ('design-data relationship', 'design-data relationship'),
        'Heatmap':
            ('representation', 'representation'),
        'It has more visual aids.':
            ('more forms', 'more forms'),
        'meaningless':
            ('no clear indication', 'no clear indication'),
        'Data correlation.':
            ('data trend', 'data trend'),
        'Design.':
            ('complex design', 'complex design'),
        'It can be quite complex to work out the different category meanings.':
            ('unclear meaning', 'unclear meaning'),
        'clarity':
            ('different clarity', 'different clarity'),
        'simplicity':
            ('simple visual elements', 'simple visual elements'),
        'Number of parameters':
            ('multiple aspects', 'multiple aspects'),
        'integration of text, numbers, and visuals all being used to convey information':
            ('technical information encoding', 'technical encoding'),
        'Understanding what the visualization is about':
            ('unclear what data conveys', 'unclear what data conveys'),
        'The level of information':
            ('more nuance information', 'more nuance information'),
        'Information about what the visualisation is showing':
            ('unclear what data conveys', 'unclear what data conveys'),
        'Understanding of what I am looking at':
            ('unclear meaning', 'unclear meaning'),
        'Understanding the shapes':
            ('shape hard to read', 'understand/read shapes'),
        'Understanding the information':
            ('unclear what data conveys', 'unclear what data conveys'),
        'Human interpretability':
            ('unclear meaning', 'unclear meaning'),
        'Able to imagine a meaning':
            ('unclear meaning', 'unclear meaning'),
        'Range of colours, presumably conveying meaning.':
            ('code meaning', 'unclear meaning'),
        'Data interpretation':
            ('unclear what data conveys', 'unclear what data conveys'),
        'Spread of information on the image':
            ('spatial organization', 'spatial organization'),
        'Volume of information':
            ('several elements', 'several elements'),
        'Color Variability and Saturation':
            ('shapes and contrasts', 'shapes and contrasts'),
        'Variety of Shapes and Patterns':
            ('variety of shapes', 'shape variety'),
        'Interconnectedness of Elements':
            ('confusing connection', 'confusing connection'),
        'Variety and Density of Elements':
            ('many shapes', 'multiple shapes'),
        'Level of Detail':
            ('fine detail', 'fine/layered details'),
        'Depth and Dimensionality':
            ('number of levels', 'number of levels'),
    },

    'Semantics / Text Legibility': {
        'How much I had to read.':
            ('a lot of text', 'amount of words/context/numbers'),
        'Less words':
            ('less text', 'little/less text/numbers/context'),
        'Meaning of titles':
            ('title', 'title/axis/label/descriptions'),
        'Use of alpha and beta':
            ('codes', 'code/text'),
        'unfamiliar terms':
            ('random meaningless names', 'lack of meaning'),
        'integration of text, numbers, and visuals all being used to convey information':
            ('text and information', 'text and imagery'),
        'font sizes, italicization, bold':
            ('different font sizes', 'different font/word sizes/structure'),
        'The use of words rather than dots makes it more complex.':
            ('a lot of words', 'amount of words/context/numbers'),
        'The additional layer of words adds complexity.':
            ('more words', 'more texts/words'),
        'It\'s more difficult to read the words because they are small;':
            ('small text', 'word rotation/small font size'),
        'Reading the values':
            ('numerical values', 'involves numbers'),
        'Comparing different values':
            ('numerical values', 'involves numbers'),
        'signs':
            ('labels', 'title/axis/label/descriptions'),
        'Mix of images and words':
            ('text and imagery', 'text and imagery'),
    },

    'Schema': {
        'Seems more mathematical':
            ('scientific method', 'a specific domain'),
        'The info doesn\'t seem to follow a specific pattern.':
            ('no pattern', 'unfamiliar concepts/patterns'),
        'The info is more varied.':
            ('information nature', 'unfamiliar concepts/patterns'),
        'Context.':
            ('no context', 'no context or reference'),
        'Concept.':
            ('familiar domain', 'domain-specific concepts (e.g., chemical, biology, map)'),
        'generated in a 3D program by mistake':
            ('3D visualization', '2D/3D'),
        'Human interpretability':
            ('no idea what it represents', 'not enough knowledge to tell'),
        'Able to imagine a meaning':
            ('no idea what it represents', 'not enough knowledge to tell'),
    },

    'Color, Symbol, and Texture Details': {
        'Level of Detail':
            ('fine details', 'color details'),
        'Depth and Dimensionality':
            ('color dimension', 'color dimension'),
        'more colours':
            ('more colors', 'more colors'),
        'colors are easy on the eyes':
            ('appealing colors', 'appealing colors'),
        'more variation':
            ('color variation', 'color variety/arrangement/distribution'),
        'The imagery is more intense in the first.':
            ('intense colors', 'intense/too many colors'),
        'It has a key colour':
            ('color key', 'color keys'),
        'numbers of unique shapes/colors':
            ('number of colors', 'amount of / too many colors'),
        'Number of colours':
            ('number of colors', 'amount of / too many colors'),
        'amount of colour used':
            ('amount of colors', 'amount of / too many colors'),
        'The colors blends':
            ('mixed colors', 'soft/layered/overlapping colors'),
        'the meaning of the colors is unclear;':
            ('unclear color meaning', 'unclear color meaning'),
        'the level of details that it has provide way to much information':
            ('too many colors', 'intense/too many colors'),
        'Numbers of colors':
            ('number of colors', 'amount of / too many colors'),
        'The image contains a greater variety of colours.':
            ('color variety', 'color variety/arrangement/distribution'),
        'the color are varied':
            ('color variation', 'color variety/arrangement/distribution'),
        'Bad color choice in regard to contrasts':
            ('poor color choice', 'low color contrast/separation/segmentation'),
        'Colour variation':
            ('color variation', 'color variety/arrangement/distribution'),
        'Range of colours, presumably conveying meaning.':
            ('colors have meaning', 'color represents groups'),
        'Size':
            ('different sizes', 'indicating symbols'),
        'pattern':
            ('contrasting patterns', 'texture details'),
        'shape':
            ('colored shape', 'colored shape'),
        'Details':
            ('color details', 'color details'),
        'Color Variability and Saturation':
            ('color saturation', 'color saturation'),
        'Variety of Shapes and Patterns':
            ('symbols', 'symbols'),
    },

    'Aesthetics Uncertainty': {
        'unknow':
            ('uncertain', 'distracting/confusing/unclear'),
        'meaningless':
            ('unclear', 'distracting/confusing/unclear'),
        'how organised/disorganised was the picture':
            ('less structured', 'looks random/messy/lack structure'),
        'Interconnectedness of Elements':
            ('convoluted composition', 'convoluted composition'),
        'looks random':
            ('messy', 'looks random/messy/lack structure'),
        'i couldnt say what is complexer':
            ('uncertain', 'distracting/confusing/unclear'),
    },
}

# ── Apply NEW_MAPPINGS into existing_lookup ─────────────────────────────────────
n_filled = 0
for topic, phrase_map in NEW_MAPPINGS.items():
    for phrase, (ai, hc) in phrase_map.items():
        key = (topic, phrase)
        if key not in existing_lookup:
            # Determine sentiment from extracted data
            sents = topic_phrase_sentiments.get(key, set())
            sent = ''
            if '(+)' in sents and '(-)' in sents:
                sent = '(+) (-)'
            elif '(+)' in sents:
                sent = '(+)'
            elif '(-)' in sents:
                sent = '(-)'
            existing_lookup[key] = {'ai': ai, 'hc': hc, 'sentiment': sent}
            n_filled += 1

# Verify all unmapped are now covered
still_unmapped = [(t, p) for (t, p) in topic_phrase_images if (t, p) not in existing_lookup]
print(f'Filled {n_filled} new AI/HC mappings')
print(f'Still unmapped: {len(still_unmapped)}')
if still_unmapped:
    for t, p in sorted(still_unmapped):
        print(f'  [{t}] "{p}"')

In [ ]:
# ── Build per-topic phrase rows ─────────────────────────────────────────────────
topic_data = {}  # topic → list of dicts

for topic in TOPICS:
    topic_phrases = {p: imgs for (t, p), imgs in topic_phrase_images.items() if t == topic}
    rows = []
    for phrase, imgs in sorted(topic_phrases.items(), key=lambda x: -len(x[1])):
        lookup = existing_lookup.get((topic, phrase), {})
        # Determine sentiment: prefer existing, else use extracted
        sent = lookup.get('sentiment', '')
        if not sent:
            sents = topic_phrase_sentiments.get((topic, phrase), set())
            if '(+)' in sents and '(-)' in sents:
                sent = '(+) (-)'
            elif '(+)' in sents:
                sent = '(+)'
            elif '(-)' in sents:
                sent = '(-)'
        rows.append({
            'original': phrase,
            'sentiment': sent,
            'count': len(imgs),
            'AIcurated': lookup.get('ai', ''),
            'HumanCurated': lookup.get('hc', ''),
        })
    topic_data[topic] = rows
    n_new = sum(1 for r in rows if not r['AIcurated'] and not r['HumanCurated'])
    print(f'{topic}: {len(rows)} phrases ({n_new} new without AI/HC mapping)')

# ── Assemble into wide-format table matching sheet 1166121277 layout ────────────
max_rows = max(len(v) for v in topic_data.values())
print(f'\nMax phrases per topic: {max_rows}')

result = {}
for topic in TOPICS:
    rows = topic_data[topic]
    padded = rows + [{'original': '', 'sentiment': '', 'count': '', 'AIcurated': '', 'HumanCurated': ''}] * (max_rows - len(rows))
    result[f'{topic}_sentiment']        = [r['sentiment'] for r in padded]
    result[f'{topic}_count']            = [r['count'] for r in padded]
    result[topic]                       = [r['original'] for r in padded]
    result[f'{topic} (AIcurated)']      = [r['AIcurated'] for r in padded]
    result[f'{topic} (HumanCuration)']  = [r['HumanCurated'] for r in padded]

df_new = pd.DataFrame(result)
print(f'New table shape: {df_new.shape}')
df_new.head(5)

## Step 4: Comparison with Existing Sheet

In [ ]:
# ── Compare old vs new table ────────────────────────────────────────────────────
print(f'Existing sheet 1166121277:  {df_existing.shape[0]} rows × {df_existing.shape[1]} cols')
print(f'New regenerated table:      {df_new.shape[0]} rows × {df_new.shape[1]} cols')

for topic in TOPICS:
    n_old = df_existing[topic].dropna().astype(str).str.strip().ne('').sum()
    n_new_t = len(topic_data[topic])
    old_set = set(df_existing[topic].dropna().astype(str).str.strip()) - {''}
    new_set = set(r['original'] for r in topic_data[topic])
    added = new_set - old_set
    removed = old_set - new_set
    print(f'\n{topic}:')
    print(f'  Old: {n_old} phrases → New: {n_new_t} phrases (Δ {n_new_t - n_old:+d})')
    print(f'  Added: {len(added)}, Removed: {len(removed)}')
    if added:
        for p in sorted(added)[:5]:
            print(f'    + "{p}"')
        if len(added) > 5:
            print(f'    ... and {len(added)-5} more')

# Total unique images
all_more = df_pairs[IMG_MORE_COL].dropna().astype(str).str.strip()
all_less = df_pairs[IMG_LESS_COL].dropna().astype(str).str.strip()
all_imgs_sheet = set(all_more[all_more != '']) | set(all_less[all_less != ''])
all_images = set()
for (t, p), imgs in topic_phrase_images.items():
    all_images |= imgs
print(f'\nTotal unique images in sheet 1103808983: {len(all_imgs_sheet)}')
print(f'Unique images with curated phrases in new table: {len(all_images)}')

## Save

In [ ]:
import os
os.makedirs('comment_process', exist_ok=True)

out_path = 'data_processing/term_mapping_regenerated.csv'
df_new.to_csv(out_path, index=False)
print(f'Saved → {out_path}  ({df_new.shape[0]} rows × {df_new.shape[1]} cols)')

## Step 5: Convert to Sheet 1160817631 Format

Convert `comment_process/term_mapping_regenerated.csv` (365-row wide format) into a long pivot table matching the layout of sheet `1160817631`:

| topic | phrase | sentiment | num_images | Area | Bar | Cont.-ColorPatn | Glyph | Grid | Line | Node-link | Point | Text |
|-------|--------|-----------|------------|------|-----|-----------------|-------|------|------|-----------|-------|------|

- **phrase** = `{topic} (HumanCuration)` column — multiple originals sharing the same HC label are aggregated (image sets unioned per topic).
- **num_images** = total images associated with the aggregated HC phrase.
- Per-VisType counts use an image→VisType lookup from the per-image sheet (`1390591889`).

In [ ]:
import pandas as pd
from collections import defaultdict

VisTypes = ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']

# ── Image → VisType lookup from per-image sheet ─────────────────────────────────
df_img_sheet = load_sheet(SPREADSHEET_PAIRS, '1390591889')
image_vistype = dict(zip(df_img_sheet['ImageName'], df_img_sheet['VisType']))

# Supplement with VisType from paired-comparison sheet (1103808983) for any missing images
for _, row in df_pairs.iterrows():
    img_more = str(row.get(IMG_MORE_COL, '')).strip()
    img_less = str(row.get(IMG_LESS_COL, '')).strip()
    vt_more  = str(row.get('VisTypesMoreComplexImageType', '')).strip()
    vt_less  = str(row.get('VisTypesLessComplexImageType', '')).strip()
    if img_more and img_more not in image_vistype and vt_more:
        image_vistype[img_more] = vt_more
    if img_less and img_less not in image_vistype and vt_less:
        image_vistype[img_less] = vt_less

print(f'VisType lookup: {len(image_vistype)} images (sheet 1390591889 + sheet 1103808983 fallback)')

# ── Load term_mapping_regenerated.csv (365 rows, wide format) ────────────────────
df_regen = pd.read_csv('data_processing/term_mapping_regenerated.csv')
print(f'Loaded term_mapping_regenerated: {df_regen.shape[0]} rows × {df_regen.shape[1]} cols')

# ── Aggregate images per (topic, hc_phrase) ──────────────────────────────────────
# Multiple original phrases may share the same HC label within a topic;
# union their image sets to avoid double-counting.
hc_images     = defaultdict(set)   # (topic, hc_phrase) → union of image names
hc_sentiments = defaultdict(set)   # (topic, hc_phrase) → set of sentiments

for _, row in df_regen.iterrows():
    for topic in TOPICS:
        original = row.get(topic, '')
        if not isinstance(original, str) or not original.strip():
            continue
        hc   = str(row.get(f'{topic} (HumanCuration)', '')).strip()
        phrase = hc if hc else original
        sent   = str(row.get(f'{topic}_sentiment', '')).strip()

        imgs = topic_phrase_images.get((topic, original), set())
        hc_images[(topic, phrase)] |= imgs
        if sent:
            for token in ['(+)', '(-)']:
                if token in sent:
                    hc_sentiments[(topic, phrase)].add(token)

# ── Build long-format table ──────────────────────────────────────────────────────
records = []
for (topic, phrase), imgs in hc_images.items():
    if not imgs:
        continue
    sents = hc_sentiments.get((topic, phrase), set())
    if '(+)' in sents and '(-)' in sents:
        sentiment = '(+) (-)'
    elif '(+)' in sents:
        sentiment = '(+)'
    elif '(-)' in sents:
        sentiment = '(-)'
    else:
        sentiment = ''

    vt_counts = {vt: 0 for vt in VisTypes}
    for img in imgs:
        vt = image_vistype.get(img, '')
        if vt in vt_counts:
            vt_counts[vt] += 1

    rec = {'topic': topic, 'phrase': phrase, 'sentiment': sentiment, 'num_images': len(imgs)}
    rec.update(vt_counts)
    records.append(rec)

df_sheet1 = (
    pd.DataFrame(records, columns=['topic', 'phrase', 'sentiment', 'num_images'] + VisTypes)
    .sort_values(['topic', 'phrase'])
    .reset_index(drop=True)
)

# ── Summary ──────────────────────────────────────────────────────────────────────
all_imgs = set()
for imgs in hc_images.values():
    all_imgs |= imgs
print(f'Total unique images: {len(all_imgs)}')
print(f'\nSheet 1160817631 format: {df_sheet1.shape[0]} rows × {df_sheet1.shape[1]} cols')
print(f'Unique HC phrases: {df_sheet1["phrase"].nunique()}')
print(f'Topics present:    {df_sheet1["topic"].nunique()}')

# Save
out_path = 'data_processing/phrase_pivot_regenerated.csv'
df_sheet1.to_csv(out_path, index=False)
print(f'\nSaved → {out_path}')
df_sheet1.head(10)